# Tutorial de Scikit-Learn: Clasificación Binaria
Basado en el tutorial de **coursera's IBM Python for Data Science and AI y notas del Dr. Ivan Olier-Caparroso**

## Introducción
En este notebook aprenderás a resolver problemas de **clasificación binaria**: predecir si una observación pertenece a una de dos clases (Sí/No, 0/1, Positivo/Negativo).

Cubriremos:
- Qué es la **Regresión Logística** y cómo funciona
- **Estratificación**: por qué es crítica en datasets desbalanceados
- **Todas las métricas de clasificación**: Accuracy, Precision, Recall, F1, ROC-AUC
- La **Matriz de Confusión** y su interpretación
- La **Curva ROC** y cómo elegir el umbral óptimo

<hr/>
<div class="alert alert-success alertsuccess" style="margin-top: 20px">
[Consejo]: Ejecuta cada celda con <kbd>Shift</kbd> + <kbd>Enter</kbd> en orden.
</div>
<hr/>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Módulos de sklearn para clasificación
from sklearn.model_selection  import train_test_split
from sklearn.linear_model     import LogisticRegression
from sklearn.preprocessing    import StandardScaler
from sklearn.preprocessing    import OneHotEncoder
from sklearn.metrics          import (
    confusion_matrix,          # Matriz de confusión
    classification_report,     # Reporte completo de métricas
    accuracy_score,            # Exactitud global
    precision_score,           # Precisión
    recall_score,              # Sensibilidad / Recall
    f1_score,                  # F1-Score
    roc_auc_score,             # Área bajo la curva ROC
    roc_curve,                 # Curva ROC completa
    precision_recall_curve     # Curva Precision-Recall
)

sns.set_theme(style='whitegrid', font_scale=1.1)
np.random.seed(42)

import sklearn
print('scikit-learn:', sklearn.__version__)

## 1. Dataset: Predicción de Abandono de Clientes (Churn)

Trabajaremos con un dataset simulado de una empresa de telecomunicaciones. El objetivo es predecir si un cliente **abandonará el servicio** (`churn = 1`) o **permanecerá** (`churn = 0`).

Este es un problema típico de clasificación binaria **desbalanceada**: en la realidad, solo un 20-30% de los clientes se dan de baja.

In [ ]:
np.random.seed(42)
n = 1000

# Variables del cliente
antiguedad_meses  = np.random.randint(1, 72, n)         # Meses como cliente
cargo_mensual     = np.random.normal(65, 20, n).clip(20, 120)  # USD/mes
num_productos     = np.random.randint(1, 5, n)          # Productos contratados
llamadas_soporte  = np.random.randint(0, 10, n)         # Llamadas al soporte
tiene_pareja      = np.random.choice([0, 1], n)         # 1 = sí tiene pareja
tipo_contrato     = np.random.choice(
    ['Mensual', 'Anual', 'Bianual'], n, p=[0.5, 0.3, 0.2]
)
metodo_pago       = np.random.choice(
    ['Tarjeta', 'Transferencia', 'Efectivo'], n, p=[0.4, 0.4, 0.2]
)

# Probabilidad de churn (función logística sobre los factores)
log_odds = (
    -2.5
    - antiguedad_meses  * 0.04   # más antiguo → menos probable que se vaya
    + cargo_mensual     * 0.025  # más caro → más probable que se vaya
    + llamadas_soporte  * 0.3    # más llamadas al soporte → más probable
    - num_productos     * 0.4    # más productos → más comprometido
    + (tipo_contrato == 'Mensual')  * 1.5  # contrato mensual → más riesgo
    - (tipo_contrato == 'Bianual')  * 1.0  # bianual → menos riesgo
)
prob_churn = 1 / (1 + np.exp(-log_odds))  # función sigmoide
churn = (np.random.uniform(0, 1, n) < prob_churn).astype(int)

df = pd.DataFrame({
    'antiguedad_meses': antiguedad_meses,
    'cargo_mensual':    cargo_mensual,
    'num_productos':    num_productos,
    'llamadas_soporte': llamadas_soporte,
    'tiene_pareja':     tiene_pareja,
    'tipo_contrato':    tipo_contrato,
    'metodo_pago':      metodo_pago,
    'churn':            churn
})

print('Forma del dataset:', df.shape)
print('\nDistribución de la variable objetivo:')
print(df['churn'].value_counts())
print(f'\n  Tasa de Churn: {df["churn"].mean()*100:.1f}% → dataset DESBALANCEADO')

In [ ]:
# Exploración visual de las variables más relevantes
# Convertimos churn a string para evitar conflictos de tipo con palette
df['churn_label'] = df['churn'].map({0: 'No Churn', 1: 'Churn'})
paleta_churn = {'No Churn': 'steelblue', 'Churn': 'tomato'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(data=df, x='antiguedad_meses', hue='churn_label', kde=True,
             ax=axes[0,0], palette=paleta_churn)
axes[0,0].set_title('Antigüedad por Churn')

sns.histplot(data=df, x='cargo_mensual', hue='churn_label', kde=True,
             ax=axes[0,1], palette=paleta_churn)
axes[0,1].set_title('Cargo Mensual por Churn')

sns.countplot(data=df, x='tipo_contrato', hue='churn_label',
              ax=axes[1,0], palette=paleta_churn)
axes[1,0].set_title('Tipo de Contrato por Churn')

# boxplot usa x categórico directamente, no necesita palette dict
sns.boxplot(data=df, x='churn_label', y='llamadas_soporte',
            ax=axes[1,1], palette=paleta_churn,
            order=['No Churn', 'Churn'])
axes[1,1].set_title('Llamadas al Soporte por Churn')
axes[1,1].set_xlabel('Churn')

plt.suptitle('Análisis Exploratorio: Variables vs Churn', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Estratificación en la Partición de Datos

### ¿Qué es la estratificación?

Cuando hacemos `train_test_split` de forma aleatoria simple, **existe el riesgo** de que la distribución de clases en cada subconjunto sea diferente a la del dataset original. Esto es especialmente problemático con datasets desbalanceados.

La **partición estratificada** garantiza que la proporción de cada clase se mantenga constante en Train, Validation y Test.

```
Dataset original: 75% No-Churn / 25% Churn

Sin estratificar (aleatorio):      Con estratificar (stratify=y):
  Train: 80% / 20%  ← distinto!     Train: 75% / 25%  ← igual ✓
  Val:   70% / 30%  ← distinto!     Val:   75% / 25%  ← igual ✓
  Test:  65% / 35%  ← distinto!     Test:  75% / 25%  ← igual ✓
```

In [ ]:
# ── Preparar features ────────────────────────────────────────────────────────
columnas_num = ['antiguedad_meses', 'cargo_mensual', 'num_productos',
                'llamadas_soporte', 'tiene_pareja']
columnas_cat = ['tipo_contrato', 'metodo_pago']

X = df[columnas_num + columnas_cat]
y = df['churn']

print(f'Proporción de churn en el dataset completo: {y.mean():.3f} ({y.mean()*100:.1f}%)')

In [ ]:
# ── Partición SIN estratificación ───────────────────────────────────────────
X_tr_no, X_te_no, y_tr_no, y_te_no = train_test_split(
    X, y, test_size=0.2, random_state=42
    # sin stratify → solo aleatoriedad
)

# ── Partición CON estratificación ───────────────────────────────────────────
# stratify=y → asegura que cada subconjunto mantenga la proporción de clases
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
)

print('═══ Comparación de proporciones de Churn ════')
print(f'Dataset completo:      {y.mean():.3f}')
print()
print('Sin estratificar:')
print(f'  Train: {y_tr_no.mean():.3f}   Test: {y_te_no.mean():.3f}  ← pueden diferir')
print()
print('Con estratificar (stratify=y):')
print(f'  Train: {y_train.mean():.3f}   Val: {y_val.mean():.3f}   Test: {y_test.mean():.3f}  ← todas iguales ✓')

In [ ]:
# Visualizar el efecto de la estratificación
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sin estratificar
proporciones_no = {
    'Completo': y.mean(),
    'Train\n(sin strat)': y_tr_no.mean(),
    'Test\n(sin strat)': y_te_no.mean()
}
axes[0].bar(proporciones_no.keys(), proporciones_no.values(), color=['gray','tomato','tomato'])
axes[0].axhline(y.mean(), color='black', linestyle='--', linewidth=1.5, label='Proporción original')
axes[0].set_title('Sin Estratificación')
axes[0].set_ylabel('Proporción de Churn')
axes[0].set_ylim(0, 0.6)
axes[0].legend()

# Con estratificar
proporciones_si = {
    'Completo': y.mean(),
    'Train\n(stratify)': y_train.mean(),
    'Val\n(stratify)': y_val.mean(),
    'Test\n(stratify)': y_test.mean()
}
axes[1].bar(proporciones_si.keys(), proporciones_si.values(), color=['gray','steelblue','steelblue','steelblue'])
axes[1].axhline(y.mean(), color='black', linestyle='--', linewidth=1.5, label='Proporción original')
axes[1].set_title('Con Estratificación (stratify=y)')
axes[1].set_ylabel('Proporción de Churn')
axes[1].set_ylim(0, 0.6)
axes[1].legend()

plt.suptitle('Efecto de la Estratificación en la Distribución de Clases', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Regresión Logística

### ¿Qué es la Regresión Logística?

A pesar de su nombre, es un algoritmo de **clasificación**. Modela la **probabilidad** de que una observación pertenezca a la clase positiva (1):

$$P(y=1 \mid x) = \sigma(\beta_0 + \beta_1 x_1 + \ldots + \beta_p x_p) = \frac{1}{1 + e^{-(\beta_0 + \boldsymbol{\beta}^T \mathbf{x})}}$$

La función $\sigma$ (sigmoide) garantiza que la salida esté **entre 0 y 1**. Si $P > 0.5$, el modelo predice clase 1; si no, clase 0.

In [ ]:
# Visualizar la función sigmoide
z = np.linspace(-7, 7, 200)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(9, 4))
plt.plot(z, sigmoid, color='steelblue', linewidth=2.5)
plt.axhline(0.5, color='red', linestyle='--', linewidth=1.5, label='Umbral = 0.5')
plt.axvline(0,   color='gray', linestyle=':', linewidth=1)
plt.fill_between(z, sigmoid, 0.5, where=(sigmoid > 0.5), alpha=0.15, color='green', label='Predice clase 1')
plt.fill_between(z, sigmoid, 0.5, where=(sigmoid < 0.5), alpha=0.15, color='red',   label='Predice clase 0')
plt.title('Función Sigmoide: convierte log-odds en probabilidad')
plt.xlabel('z = β₀ + β₁x₁ + ... (combinación lineal)')
plt.ylabel('P(y=1 | x)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Preparar features: OHE para categóricas + escalado ──────────────────────

# OHE para variables categóricas
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
encoder.fit(X_train[columnas_cat])

def preparar(X_part, encoder, columnas_num, columnas_cat):
    enc = encoder.transform(X_part[columnas_cat])
    nombres = encoder.get_feature_names_out(columnas_cat)
    df_enc = pd.DataFrame(enc, columns=nombres, index=X_part.index)
    return pd.concat([X_part[columnas_num].reset_index(drop=True),
                      df_enc.reset_index(drop=True)], axis=1)

X_train_enc = preparar(X_train, encoder, columnas_num, columnas_cat)
X_val_enc   = preparar(X_val,   encoder, columnas_num, columnas_cat)
X_test_enc  = preparar(X_test,  encoder, columnas_num, columnas_cat)

# Estandarización: importante para regresión logística
# fit SOLO en train para evitar data leakage
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_enc)
X_val_sc   = scaler.transform(X_val_enc)
X_test_sc  = scaler.transform(X_test_enc)

print('Features finales:', X_train_enc.columns.tolist())
print('Forma X_train escalado:', X_train_sc.shape)

In [ ]:
# ── Entrenar Regresión Logística ─────────────────────────────────────────────
# max_iter=1000 → número máximo de iteraciones del optimizador
# C=1.0        → inverso de la regularización (mayor C = menos regularización)
# solver='lbfgs' → algoritmo de optimización (bueno para datasets medianos)

modelo_lr = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', random_state=42)
modelo_lr.fit(X_train_sc, y_train)

# Predicciones: clase (0 ó 1) y probabilidades
y_pred_val  = modelo_lr.predict(X_val_sc)          # Clase predicha
y_prob_val  = modelo_lr.predict_proba(X_val_sc)[:, 1]  # P(churn=1)

print('Primeras 10 predicciones:')
print('  Clase real:    ', y_val.values[:10])
print('  Clase predicha:', y_pred_val[:10])
print('  P(churn=1):    ', y_prob_val[:10].round(3))

## 4. Métricas de Clasificación

### 4.1 La Matriz de Confusión

Es la base de todas las métricas. Cruza las clases **reales** con las **predichas**:

```
                    PREDICHO
                  No-Churn  Churn
REAL  No-Churn  [   TN    |  FP  ]   TN = True Negative  (acierto en clase 0)
      Churn     [   FN    |  TP  ]   TP = True Positive  (acierto en clase 1)
                                     FP = False Positive (predijo 1, era 0) → Error Tipo I
                                     FN = False Negative (predijo 0, era 1) → Error Tipo II
```

In [ ]:
def graficar_matriz_confusion(y_real, y_pred, titulo='Matriz de Confusión'):
    cm = confusion_matrix(y_real, y_pred)
    tn, fp, fn, tp = cm.ravel()

    etiquetas = np.array([
        [f'TN\n{tn}', f'FP\n{fp}'],
        [f'FN\n{fn}', f'TP\n{tp}']
    ])

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=etiquetas, fmt='', cmap='Blues',
                xticklabels=['No-Churn (0)', 'Churn (1)'],
                yticklabels=['No-Churn (0)', 'Churn (1)'],
                linewidths=1)
    plt.title(titulo)
    plt.xlabel('Predicho')
    plt.ylabel('Real')
    plt.tight_layout()
    plt.show()
    return tn, fp, fn, tp

tn, fp, fn, tp = graficar_matriz_confusion(y_val, y_pred_val)

### 4.2 Todas las Métricas Explicadas

A partir de TN, FP, FN, TP se derivan todas las métricas:

| Métrica | Fórmula | Pregunta que responde |
|---|---|---|
| **Accuracy** | $\frac{TP+TN}{TP+TN+FP+FN}$ | ¿Qué fracción de predicciones son correctas? |
| **Precision** | $\frac{TP}{TP+FP}$ | De los que predije como Churn, ¿cuántos realmente se fueron? |
| **Recall (Sensibilidad)** | $\frac{TP}{TP+FN}$ | De todos los que se fueron, ¿cuántos detecté? |
| **Specificity** | $\frac{TN}{TN+FP}$ | De los que se quedaron, ¿cuántos identifiqué correctamente? |
| **F1-Score** | $\frac{2 \cdot P \cdot R}{P+R}$ | Media armónica entre Precision y Recall |
| **ROC-AUC** | Área bajo curva ROC | Capacidad general del modelo para discriminar clases |

In [ ]:
# Calcular todas las métricas manualmente desde la matriz de confusión
accuracy    = (tp + tn) / (tp + tn + fp + fn)
precision   = tp / (tp + fp) if (tp + fp) > 0 else 0
recall      = tp / (tp + fn) if (tp + fn) > 0 else 0  # también llamado Sensibilidad
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
f1          = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
roc_auc     = roc_auc_score(y_val, y_prob_val)

print('═══ MÉTRICAS DE CLASIFICACIÓN (Validation) ══════════════')
print(f'  Accuracy    (exactitud global):    {accuracy:.4f}')
print(f'  Precision   (de mis alertas, % OK):{precision:.4f}')
print(f'  Recall      (% churns detectados): {recall:.4f}')
print(f'  Specificity (% no-churns OK):      {specificity:.4f}')
print(f'  F1-Score    (balance P y R):       {f1:.4f}')
print(f'  ROC-AUC     (discriminación):      {roc_auc:.4f}')
print()
print('Verificación con sklearn:')
print(f'  accuracy_score:   {accuracy_score(y_val, y_pred_val):.4f}')
print(f'  precision_score:  {precision_score(y_val, y_pred_val):.4f}')
print(f'  recall_score:     {recall_score(y_val, y_pred_val):.4f}')
print(f'  f1_score:         {f1_score(y_val, y_pred_val):.4f}')

In [ ]:
# Reporte completo de sklearn (incluye métricas para ambas clases)
print('─── Classification Report ──────────────────────────────────')
print(classification_report(y_val, y_pred_val, target_names=['No-Churn (0)', 'Churn (1)']))
print('Nota: support = número de muestras reales en cada clase')

### 4.3 ¿Por qué Accuracy puede ser engañosa?

En datasets desbalanceados, un modelo que **siempre predice la clase mayoritaria** puede tener una accuracy alta sin aprender nada útil.

In [ ]:
# Modelo 'dummy': siempre predice No-Churn (0)
y_dummy = np.zeros(len(y_val), dtype=int)

print('Modelo DUMMY (siempre predice 0 - No Churn):')
print(f'  Accuracy:  {accuracy_score(y_val, y_dummy):.4f}  ← parece bueno!')
print(f'  Recall:    {recall_score(y_val, y_dummy, zero_division=0):.4f}  ← detecta 0% de los churns')
print(f'  F1-Score:  {f1_score(y_val, y_dummy, zero_division=0):.4f}  ← modelo inútil')
print()
print('Modelo de Regresión Logística:')
print(f'  Accuracy:  {accuracy_score(y_val, y_pred_val):.4f}')
print(f'  Recall:    {recall_score(y_val, y_pred_val):.4f}  ← sí detecta churns')
print(f'  F1-Score:  {f1_score(y_val, y_pred_val):.4f}')
print()
print('→ En problemas desbalanceados: F1, Recall y ROC-AUC son más informativas que Accuracy')

### 4.4 Curva ROC y Elección del Umbral

La **Curva ROC** (Receiver Operating Characteristic) muestra el trade-off entre **Recall (TPR)** y **False Positive Rate (FPR)** para todos los posibles umbrales de clasificación.

- Un clasificador perfecto tiene **AUC = 1.0**
- Un clasificador aleatorio tiene **AUC = 0.5** (diagonal)
- El **umbral por defecto** es 0.5, pero puede ajustarse según el problema

In [ ]:
fpr, tpr, umbrales = roc_curve(y_val, y_prob_val)
auc = roc_auc_score(y_val, y_prob_val)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='steelblue', linewidth=2.5,
         label=f'Regresión Logística (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Clasificador aleatorio (AUC = 0.5)')
plt.fill_between(fpr, tpr, alpha=0.1, color='steelblue')

# Marcar el umbral 0.5
idx_05 = np.argmin(np.abs(umbrales - 0.5))
plt.scatter(fpr[idx_05], tpr[idx_05], color='red', s=100, zorder=5,
            label=f'Umbral=0.5 (FPR={fpr[idx_05]:.2f}, TPR={tpr[idx_05]:.2f})')

plt.xlabel('FPR (False Positive Rate = 1 - Specificity)')
plt.ylabel('TPR (True Positive Rate = Recall)')
plt.title('Curva ROC')
plt.legend()
plt.tight_layout()
plt.show()

### 4.5 Trade-off entre Precision y Recall según el Umbral

El umbral 0.5 no es siempre el óptimo. Dependiendo del problema:

- **Queremos más Recall** (detectar todos los churns, aunque haya falsas alarmas) → bajar el umbral
- **Queremos más Precision** (solo alertar cuando estemos muy seguros) → subir el umbral

In [ ]:
# Cómo cambian Precision y Recall al variar el umbral
umbrales_rango = np.linspace(0.1, 0.9, 100)
precisiones, recalls, f1s = [], [], []

for u in umbrales_rango:
    y_pred_u = (y_prob_val >= u).astype(int)
    precisiones.append(precision_score(y_val, y_pred_u, zero_division=0))
    recalls.append(recall_score(y_val, y_pred_u, zero_division=0))
    f1s.append(f1_score(y_val, y_pred_u, zero_division=0))

# Umbral óptimo según F1
idx_opt = np.argmax(f1s)
umbral_opt = umbrales_rango[idx_opt]

plt.figure(figsize=(10, 5))
plt.plot(umbrales_rango, precisiones, label='Precision', color='blue',   linewidth=2)
plt.plot(umbrales_rango, recalls,    label='Recall',    color='green',   linewidth=2)
plt.plot(umbrales_rango, f1s,        label='F1-Score',  color='orange',  linewidth=2)
plt.axvline(0.5,         color='red',   linestyle='--', linewidth=1.5, label='Umbral 0.5 (default)')
plt.axvline(umbral_opt,  color='purple',linestyle='--', linewidth=1.5,
            label=f'Umbral óptimo F1 = {umbral_opt:.2f}')
plt.xlabel('Umbral de Clasificación')
plt.ylabel('Valor de la Métrica')
plt.title('Precision, Recall y F1 según el Umbral')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Umbral óptimo por F1: {umbral_opt:.2f}')
print(f'  F1 con umbral 0.5:    {f1_score(y_val, (y_prob_val>=0.5).astype(int)):.4f}')
print(f'  F1 con umbral óptimo: {f1s[idx_opt]:.4f}')

In [ ]:
# Comparar umbral 0.5 vs umbral óptimo
y_pred_opt = (y_prob_val >= umbral_opt).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_p, titulo in zip(
    axes,
    [y_pred_val, y_pred_opt],
    [f'Umbral 0.5\nF1={f1_score(y_val,y_pred_val):.3f}',
     f'Umbral óptimo ({umbral_opt:.2f})\nF1={f1_score(y_val,y_pred_opt):.3f}']
):
    cm = confusion_matrix(y_val, y_p)
    tn_i, fp_i, fn_i, tp_i = cm.ravel()
    etiq = np.array([[f'TN\n{tn_i}', f'FP\n{fp_i}'], [f'FN\n{fn_i}', f'TP\n{tp_i}']])
    sns.heatmap(cm, annot=etiq, fmt='', cmap='Blues', ax=ax,
                xticklabels=['No-Churn', 'Churn'],
                yticklabels=['No-Churn', 'Churn'], linewidths=1)
    ax.set_title(titulo)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

plt.suptitle('Impacto del Umbral en la Matriz de Confusión', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Evaluación Final en Test

Con el modelo y umbral seleccionados, realizamos la evaluación final e **irreversible** en Test:

In [ ]:
# Predicciones en Test con el umbral óptimo
y_prob_test = modelo_lr.predict_proba(X_test_sc)[:, 1]
y_pred_test = (y_prob_test >= umbral_opt).astype(int)

print('═══ EVALUACIÓN FINAL EN TEST ════════════════════════════')
print(classification_report(y_test, y_pred_test, target_names=['No-Churn', 'Churn']))

auc_test = roc_auc_score(y_test, y_prob_test)
print(f'  ROC-AUC (Test): {auc_test:.4f}')

graficar_matriz_confusion(y_test, y_pred_test, 'Matriz de Confusión - TEST FINAL')

## 6. Resumen Visual de todas las Métricas

In [ ]:
# Dashboard resumen
metricas_finales = {
    'Accuracy':    accuracy_score(y_test, y_pred_test),
    'Precision':   precision_score(y_test, y_pred_test),
    'Recall':      recall_score(y_test, y_pred_test),
    'F1-Score':    f1_score(y_test, y_pred_test),
    'ROC-AUC':     roc_auc_score(y_test, y_prob_test)
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras de métricas
colores_bar = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
bars = axes[0].bar(metricas_finales.keys(), metricas_finales.values(), color=colores_bar)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Métricas de Clasificación - Test Final')
axes[0].set_ylabel('Valor')
for bar, val in zip(bars, metricas_finales.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

# Curva ROC en test
fpr_t, tpr_t, _ = roc_curve(y_test, y_prob_test)
axes[1].plot(fpr_t, tpr_t, color='steelblue', linewidth=2.5,
             label=f'AUC = {auc_test:.3f}')
axes[1].plot([0,1],[0,1],'r--', linewidth=1.5, label='Aleatorio')
axes[1].fill_between(fpr_t, tpr_t, alpha=0.1, color='steelblue')
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR (Recall)')
axes[1].set_title('Curva ROC - Test Final')
axes[1].legend()

plt.tight_layout()
plt.show()

## Ejercicio Final

1. **Umbral de negocio**: Supón que en este caso de churn, perder un cliente cuesta **5 veces más** que enviar una oferta de retención innecesaria. ¿Qué métrica deberías maximizar? ¿Qué umbral usarías?

2. **Sin estratificación**: Repite el entrenamiento sin usar `stratify=y` en `train_test_split`. ¿Cambian las métricas finales? ¿Cuánto?

3. **Regularización**: Prueba valores de `C = [0.01, 0.1, 1, 10, 100]` en la Regresión Logística y compara el F1 en Validation para cada uno. ¿Cuál es el mejor?

In [ ]:
# Escribe tu solución aquí


Haz doble clic **aquí** para ver pistas.

<!--
# 1. Si FN cuesta 5x más que FP → maximizar Recall
#    Bajar el umbral (ej. 0.3) aumenta Recall a costa de Precision

# 3. Grid de regularización
for C in [0.01, 0.1, 1, 10, 100]:
    m = LogisticRegression(C=C, max_iter=1000, random_state=42)
    m.fit(X_train_sc, y_train)
    f1 = f1_score(y_val, m.predict(X_val_sc))
    print(f'C={C:6.2f} → F1 Validation = {f1:.4f}')
-->

### 4.6 Coeficientes del modelo: ¿qué factores predicen el churn? Ejercicio en casa!!!

In [ ]:
# ── ¿Qué son los Log-Odds? ───────────────────────────────────────────────
# La regresión logística NO predice probabilidades directamente.
# Internamente modela los LOG-ODDS (también llamado 'logit'):
#
#   log-odds = log( P(churn=1) / P(churn=0) )
#            = log( P(evento) / P(no evento) )
#
# Es el logaritmo natural de la razón entre la probabilidad de que
# ocurra el evento vs. que NO ocurra.
#
# Ejemplos intuitivos:
#   log-odds =  0.0  →  P = 0.50  (igual de probable que sí o que no)
#   log-odds =  1.0  →  P = 0.73  (más probable que sí)
#   log-odds = -1.0  →  P = 0.27  (más probable que no)
#   log-odds = -2.5  →  P = 0.08  (muy poco probable que sí)
#
# Cada coeficiente β indica cuánto CAMBIA el log-odds por cada unidad
# de aumento en esa variable (manteniendo el resto constante).
#
# ── ¿Qué son los Odds Ratio (OR)? ────────────────────────────────────────
# El Odds Ratio es simplemente e^(log-odds) = e^β.
# Es más fácil de interpretar porque trabaja en escala multiplicativa:
#
#   OR = e^β = P(evento) / P(no evento)
#
# Interpretación directa:
#   OR = 1.0  → la variable NO tiene efecto sobre el churn
#   OR = 2.0  → esa variable DUPLICA las probabilidades de churn
#   OR = 0.5  → esa variable REDUCE A LA MITAD las probabilidades de churn
#   OR = 3.5  → multiplica por 3.5 las probabilidades de churn
#
# Ejemplo concreto con llamadas_soporte (β ≈ 0.3):
#   OR = e^0.3 ≈ 1.35
#   → Cada llamada adicional al soporte multiplica por 1.35
#     las probabilidades de que el cliente se vaya.

nombres_features = (columnas_num +
                    encoder.get_feature_names_out(columnas_cat).tolist())

coef_df = pd.DataFrame({
    'Feature':                 nombres_features,
    'Coeficiente (log-odds)':  modelo_lr.coef_[0],        # β: escala log
    'Odds Ratio (OR)':         np.exp(modelo_lr.coef_[0]) # e^β: escala multiplicativa
}).sort_values('Coeficiente (log-odds)')

print(coef_df.to_string(index=False))
print()
print('Interpretación del Odds Ratio:')
print('  OR > 1 → esa variable AUMENTA la probabilidad de churn')
print('  OR = 1 → esa variable NO tiene efecto')
print('  OR < 1 → esa variable REDUCE  la probabilidad de churn')
print()
print('Interpretación del Coeficiente (log-odds):')
print('  β > 0  → aumenta el log-odds → aumenta P(churn)')
print('  β = 0  → sin efecto')
print('  β < 0  → reduce  el log-odds → reduce  P(churn)')

In [ ]:
# Gráfico de coeficientes
plt.figure(figsize=(10, 6))
colores = ['tomato' if c > 0 else 'steelblue' for c in coef_df['Coeficiente (log-odds)']]
plt.barh(coef_df['Feature'], coef_df['Coeficiente (log-odds)'], color=colores)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Coeficientes de la Regresión Logística\n(rojo = aumenta churn, azul = reduce churn)')
plt.xlabel('Coeficiente (log-odds)')
plt.tight_layout()
plt.show()

Que podemos analizar de la figura generada? 